In [1]:
import os
import random
from PIL import Image

MIN_SIZE = 161  # mayor que 160 — mínimo para MS-SSIM (4 downsamplings)

def copiar_imagenes_balanceadas(n):
    
    rutas = {
        "jpg":  (r"C:\Users\santi\Jeremias\TrabajoGrado\Imagenes\jpg_select",  "Imagenes/jpg"),
        "png":  (r"C:\Users\santi\Jeremias\TrabajoGrado\Imagenes\png_select",  "Imagenes/png"),
        "webp": (r"C:\Users\santi\Jeremias\TrabajoGrado\Imagenes\webp_select", "Imagenes/webp"),
    }
    
    for formato, (ruta_src, ruta_dst) in rutas.items():
        
        os.makedirs(ruta_dst, exist_ok=True)
        
        # ── Solo archivos del formato correcto ──
        archivos = [
            f for f in os.listdir(ruta_src)
            if os.path.isfile(os.path.join(ruta_src, f))
            and f.lower().endswith(f".{formato}")
        ]

        # ── Filtrar por tamaño válido para MS-SSIM ──
        archivos_validos = []
        descartadas = 0

        for archivo in archivos:
            ruta = os.path.join(ruta_src, archivo)
            try:
                with Image.open(ruta) as im:
                    w, h = im.size
                    if w >= MIN_SIZE and h >= MIN_SIZE:
                        archivos_validos.append(archivo)
                    else:
                        descartadas += 1
            except Exception:
                descartadas += 1  # archivo corrupto o ilegible

        print(f"[{formato}] {len(archivos_validos)} válidas — {descartadas} descartadas por tamaño")

        if len(archivos_validos) < n:
            raise ValueError(
                f"No hay suficientes imágenes .{formato} válidas en {ruta_src} "
                f"({len(archivos_validos)} disponibles, se necesitan {n})"
            )
        
        seleccion = random.sample(archivos_validos, n)
        
        for archivo in seleccion:
            src_path = os.path.join(ruta_src, archivo)
            dst_path = os.path.join(ruta_dst, archivo)
            
            if os.path.exists(dst_path):
                nombre, ext = os.path.splitext(archivo)
                dst_path = os.path.join(ruta_dst, f"{nombre}_copy{ext}")
            
            # ── Convertir a RGB y guardar ──
            with Image.open(src_path) as im:
                if im.mode == "P" and "transparency" in im.info:
                    im = im.convert("RGBA")
                im_rgb = im.convert("RGB")
            
            im_rgb.save(dst_path)

            # ── Verificar que quedó en RGB ──
            with Image.open(dst_path) as check:
                if check.mode != "RGB":
                    raise RuntimeError(f"Imagen no quedó en RGB: {dst_path} — modo: {check.mode}")
        
        print(f"[OK] {n} imágenes RGB .{formato} copiadas → {ruta_dst}")

In [9]:
copiar_imagenes_balanceadas(2018)

[jpg] 9895 válidas — 5 descartadas por tamaño
[OK] 2018 imágenes RGB .jpg copiadas → Imagenes/jpg
[png] 13417 válidas — 23 descartadas por tamaño
[OK] 2018 imágenes RGB .png copiadas → Imagenes/png
[webp] 9948 válidas — 0 descartadas por tamaño
[OK] 2018 imágenes RGB .webp copiadas → Imagenes/webp


In [3]:
import os
from PIL import Image

def verificar_imagenes_rgb(ruta_carpeta, mostrar=True):
    """
    Verifica que todas las imágenes en una carpeta sean RGB.
    
    Args:
        ruta_carpeta (str): Ruta de la carpeta con imágenes
        mostrar (bool): Si imprime las que no son RGB
    
    Returns:
        list: Lista de imágenes que NO son RGB
    """
    
    no_rgb = []
    
    for archivo in os.listdir(ruta_carpeta):
        path = os.path.join(ruta_carpeta, archivo)
        
        # Solo archivos
        if not os.path.isfile(path):
            continue
        
        try:
            with Image.open(path) as img:
                if img.mode != "RGB":
                    no_rgb.append(archivo)
                    
                    if mostrar:
                        print(f"❌ {archivo} → modo: {img.mode}")
        
        except Exception as e:
            if mostrar:
                print(f"[ERROR] {archivo}: {e}")
    
    if mostrar:
        print(f"\nTotal NO RGB: {len(no_rgb)}")
    
    return no_rgb

In [6]:
verificar_imagenes_rgb("Imagenes/jpg")
verificar_imagenes_rgb("Imagenes/png") 
verificar_imagenes_rgb("Imagenes/webp") 


Total NO RGB: 0

Total NO RGB: 0

Total NO RGB: 0


[]

In [1]:
import os
import random
import shutil

def copiar_png(n):
    
    ruta_src = r"C:\Users\santi\Jeremias\TrabajoGrado\Imagenes\png_select"
    ruta_dst = "Imagenes/png"
    
    os.makedirs(ruta_dst, exist_ok=True)
    
    archivos = [
        f for f in os.listdir(ruta_src)
        if os.path.isfile(os.path.join(ruta_src, f)) and f.lower().endswith(".png")
    ]
    
    if len(archivos) < n:
        raise ValueError(f"No hay suficientes imágenes PNG en {ruta_src}")
    
    seleccion = random.sample(archivos, n)
    
    for archivo in seleccion:
        src_path = os.path.join(ruta_src, archivo)
        dst_path = os.path.join(ruta_dst, archivo)
        
        # evitar sobrescribir
        if os.path.exists(dst_path):
            nombre, ext = os.path.splitext(archivo)
            dst_path = os.path.join(ruta_dst, f"{nombre}_copy{ext}")
        
        shutil.copy2(src_path, dst_path)
    
    print(f"[OK] {n} imágenes PNG copiadas a {ruta_dst}")

In [2]:
copiar_png(1500)

[OK] 1500 imágenes PNG copiadas a Imagenes/png


In [7]:
import os
import hashlib
from PIL import Image
import numpy as np

def eliminar_duplicados(carpeta: str, dry_run: bool = True):
    """
    Detecta y elimina imágenes duplicadas por contenido (no por nombre).
    
    Args:
        carpeta:  Ruta de la carpeta a limpiar.
        dry_run:  Si True, solo muestra qué se borraría sin borrar nada.
                  Ponlo en False para borrar de verdad.
    """
    extensiones = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}

    archivos = [
        os.path.join(carpeta, f)
        for f in os.listdir(carpeta)
        if os.path.isfile(os.path.join(carpeta, f))
        and os.path.splitext(f)[1].lower() in extensiones
    ]

    print(f"📁 {carpeta}")
    print(f"   {len(archivos)} imágenes encontradas\n")

    # ── Calcular hash de cada imagen ──
    hashes = {}       # hash → primer archivo encontrado (se conserva)
    duplicados = []   # archivos a eliminar

    for ruta in archivos:
        try:
            with Image.open(ruta) as im:
                # Convertir a RGB para comparación uniforme
                arr = np.array(im.convert("RGB"))
                hash_img = hashlib.md5(arr.tobytes()).hexdigest()

            if hash_img in hashes:
                duplicados.append((ruta, hashes[hash_img]))
            else:
                hashes[hash_img] = ruta

        except Exception as e:
            print(f"   ⚠ No se pudo leer: {os.path.basename(ruta)} — {e}")

    # ── Reporte ──
    print(f"   Únicas:     {len(hashes)}")
    print(f"   Duplicadas: {len(duplicados)}\n")

    if not duplicados:
        print("✅ No hay duplicados.")
        return

    for ruta_dup, ruta_orig in duplicados:
        print(f"   🗑  {os.path.basename(ruta_dup)}")
        print(f"       duplicado de → {os.path.basename(ruta_orig)}")

        if not dry_run:
            os.remove(ruta_dup)

    if dry_run:
        print(f"\n⚠ dry_run=True — no se borró nada.")
        print(f"   Llama con dry_run=False para borrar los {len(duplicados)} duplicados.")
    else:
        print(f"\n✅ {len(duplicados)} imágenes duplicadas eliminadas.")

In [8]:
eliminar_duplicados(r"C:\Users\santi\Jeremias\TrabajoGrado\Imagenes\webp_select", dry_run=False)

📁 C:\Users\santi\Jeremias\TrabajoGrado\Imagenes\webp_select
   10077 imágenes encontradas

   Únicas:     9948
   Duplicadas: 129

   🗑  blick-onlyfans-nutzung-2025-0494.webp
       duplicado de → alternatives-to-onlyfans-2025-3819.webp
   🗑  dan-benson-onlyfans-7207.webp
       duplicado de → colt-onlyfans-90-day-fiance-leaked-rumors.webp
   🗑  erika-90-day-fiance-onlyfans-leaked-trending-leaked-onlyfans-buzz.webp
       duplicado de → egirl-onlyfans-lgbtq-3359.webp
   🗑  faceless-onlyfans-creator-guide-5045.webp
       duplicado de → colt-onlyfans-90-day-fiance-leaked-rumors.webp
   🗑  find-lgbtq-onlyfans-creators-1872.webp
       duplicado de → famous-onlyfans-creators-inside-scoop-0277.webp
   🗑  lgbtq-onlyfans-management-7629.webp
       duplicado de → colt-90-day-fiance-onlyfans-leaked-leaked-onlyfans-trends.webp
   🗑  male-onlyfans-creator-gay-trend-2924.webp
       duplicado de → find-someones-onlyfans-4734.webp
   🗑  male-onlyfans-earnings-9788.webp
       duplicado de → leake